In [8]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType

spark = SparkSession.builder.appName("CNPJ_ingestion").getOrCreate()

In [9]:
EMPRESA_COLUMNS = [
    "cnpj_basico", "razao_social", "natureza_juridica", "qualificacao_responsavel",
    "capital_social", "porte", "ente_federativo_responsavel"
]
ESTABELE_COLUMNS = [
        "cnpj_basico",
        "cnpj_ordem",
        "cnpj_dv",
        "identificador_matriz_filial",
        "nome_fantasia",
        "situacao_cadastral",
        "data_situacao_cadastral",
        "motivo_situacao_cadastral",
        "nome_cidade_exterior",
        "pais",
        "data_inicio_atividade",
        "cnae_fiscal_principal",
        "cnae_fiscal_secundaria",
        "tipo_logradouro",
        "logradouro",
        "numero",
        "complemento",
        "bairro",
        "cep",
        "uf",
        "municipio",
        "ddd_1",
        "telefone_1",
        "ddd_2",
        "telefone_2",
        "ddd_fax",
        "fax",
        "correio_eletronico",
        "situacao_especial",
        "data_situacao_especial"
    ]
EMPRESA_SCHEMA = StructType([StructField(c, StringType(), True) for c in EMPRESA_COLUMNS])
ESTABELE_SCHEMA = StructType([StructField(c, StringType(), True) for c in ESTABELE_COLUMNS])

CNAE_SCHEMA = StructType([
    StructField("codigo", StringType(), False),
    StructField("descricao", StringType(), True)
])



In [ ]:
# %sql
# use catalog `workspace`; select * from `database_saraivx`.`bronze_cnpj_data` limit 100;

> Define the Extraction function for CNPJ data.

In [10]:
def ingest_cnpj_df(path, cols, sep = ";", encoding = "ISO-8859-1"):
    return (
    spark.read
        .format("csv")
        .option("delimiter", sep)
        .option("encoding", encoding)
        .schema(cols)
        .load(path)
)

In [11]:
cnae_df = ingest_cnpj_df(
    path="./data/*CNAECSV", cols=CNAE_SCHEMA
)

empresa_df = ingest_cnpj_df(
    path="./data/*EMPRECSV", cols=EMPRESA_SCHEMA
)

estabelecimento_df = ingest_cnpj_df(
    path="./data/*ESTABELE", cols=ESTABELE_SCHEMA
)

In [12]:
empresa_df.createOrReplaceTempView("empresa")
estabelecimento_df.createOrReplaceTempView("estabelecimento")
cnae_df.createOrReplaceTempView("cnae")

In [13]:
print('empresa count:',empresa_df.count())
print('estabelecimento count:',estabelecimento_df.count())
print('cnae count:',cnae_df.count())

empresa count: 27188575
cnae count: 1359


In [ ]:
# %sql

# SELECT * FROM workspace.database_saraivx.bronze_cnpj_data

In [ ]:
%sql 
CREATE OR REPLACE TABLE workspace.database_saraivx.bronze_cnpj_data AS 

SELECT /*+ BROADCAST(c, d) */ 
  a.cnpj_basico as cnpj_basico_a,
  a.razao_social,
  a.natureza_juridica,
  a.qualificacao_responsavel,
  a.capital_social,
  a.porte,
  a.ente_federativo_responsavel,
  b.cnpj_ordem,
  b.cnpj_dv,
  b.identificador_matriz_filial,
  b.nome_fantasia,
  b.situacao_cadastral,
  b.data_situacao_cadastral,
  b.motivo_situacao_cadastral,
  b.nome_cidade_exterior,
  b.pais,
  b.data_inicio_atividade,
  b.cnae_fiscal_principal,
  b.cnae_fiscal_secundaria,
  b.tipo_logradouro,
  b.logradouro,
  b.numero,
  b.complemento,
  b.bairro,
  b.cep,
  b.uf,
  b.municipio,
  b.ddd_1,
  b.telefone_1,
  b.ddd_2,
  b.telefone_2,
  b.ddd_fax,
  b.fax,
  b.correio_eletronico,
  b.situacao_especial,
  b.data_situacao_especial,
  c.descricao AS cnae_descricao_1,
  d.descricao AS cnae_descricao_2
FROM empresa a
  LEFT JOIN estabelecimento b
    ON a.cnpj_basico = b.cnpj_basico
  LEFT JOIN cnae c 
    ON b.cnae_fiscal_principal = c.codigo
    LEFT JOIN cnae d
      ON b.cnae_fiscal_secundaria = d.codigo